# CSAT by week

Customer satisfaction with the returns assistant, week by week, for Ana's weekly pilot report and its *CSAT by week*
chart. When a conversation ends, the chat asks the customer to rate it from 1 to 5. This notebook averages the ratings
by week, Monday to Sunday, next to how many conversations there were and the share the assistant resolved without a
person, and writes them to `csat_weekly`. The report reads that table from OneLake, so there's nothing to export.

Scheduled on Thursdays at 16:00, before the report. Default lakehouse: `contoso_sales`. Owner: Javier.

In [1]:
from pyspark.sql import functions as F

PILOT_START = "2026-09-14"  # the Monday the first customers met the assistant

# One row per conversation with the assistant, from the support desk: when it ended, whether the assistant resolved
# it on its own (not if a person took over, or the customer gave up), and the rating, if the customer gave one. The
# Copy conversations pipeline lands them every hour.
conversations = spark.read.table("conversations").filter(F.col("ended_at") >= PILOT_START)

## By week

Weeks start on Monday, so the current one is partial: the report says which days it covers. `ratings` counts the
customers who answered, about a third of them.

In [2]:
weekly = (
    conversations.groupBy(F.date_trunc("week", "ended_at").cast("date").alias("week"))
    .agg(
        F.count("*").alias("conversations"),
        F.round(100 * F.avg(F.col("resolved").cast("int"))).cast("int").alias("resolved_pct"),
        F.count("rating").alias("ratings"),  # count() skips the conversations nobody rated
        F.round(F.avg("rating"), 1).alias("csat"),
    )
    .orderBy("week")
)

weekly.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("csat_weekly")

## Check

The target is 4.5. A week with fewer than 100 ratings is too few to read anything into a tenth: say so in the report
instead of showing it.

In [3]:
spark.read.table("csat_weekly").orderBy("week").show()

+----------+-------------+------------+-------+----+
|      week|conversations|resolved_pct|ratings|csat|
+----------+-------------+------------+-------+----+
|2026-09-14|          642|          41|    205| 4.1|
|2026-09-21|          871|          49|    283| 4.2|
|2026-09-28|         1043|          55|    344| 4.3|
|2026-10-05|         1398|          62|    468| 4.5|
|2026-10-12|         1186|          67|    401| 4.6|
+----------+-------------+------------+-------+----+
